# 5 — Optimize a primitive parameter

> **In 3–5 minutes**
>
> **Learn:** distinguish a component parameter from an active search
> variable. **Run:** optimize the primitive resonator capacitor.
> **Inspect:** variables, bounds, normalization, and the best
> ParameterSet. **Status:** `CONVERGING` scaffold; compute wait
> excluded.

## Declare the search, not a new circuit

The Plan’s `ParameterRef` identifies a bindable physical value. Only
`OptimizationVariable` activates it and supplies finite bounds. This
separation is the [parameter-versus-variable
contract](../../docs/concepts/units-parameters-and-optimization.qmd#parameter-vs-optimization-variable);
it does not require a `CompositePlan`.

`CostObjective` compares a typed quantity with a target after relative
normalization. `CMAESSpec` supplies execution controls, and
`OptimizationSpec` binds the variables, objectives, and optimizer into
one inspectable request.

In [ ]:
from fixtures.primitive_resonator import build_primitive_resonator
from scnsim import (
    CMAESSpec,
    CircuitRun,
    CostObjective,
    DiagonalRootSpec,
    OptimizationSpec,
    OptimizationVariable,
    ReductionPipeline,
    units as u,
)

fixture = build_primitive_resonator()
run = CircuitRun(plan=fixture.plan, workspace="workspaces/primitive-course")
view = run.original.reduce(ReductionPipeline().retain(fixture.resonator_node))
root_spec = DiagonalRootSpec(
    coordinate=fixture.resonator_node,
    root_hint=6.0 * u.GHz,
)
optimization_spec = OptimizationSpec(
    variables=(
        OptimizationVariable(
            parameter=fixture.resonator_capacitance,
            bounds=(80.0 * u.fF, 140.0 * u.fF),
        ),
    ),
    objectives=(
        CostObjective(
            id="resonance_frequency",
            quantity=root_spec.frequency,
            target=6.2 * u.GHz,
            weight=1.0 * u.dimensionless,
        ),
    ),
    optimizer=CMAESSpec(seed=17, max_evaluations=200),
)
optimization_spec.show()

## Execute once and keep the winner explicit

Candidate binding, quantity evaluation, objective aggregation, and
CMA-ES stay inside one Julia process. The returned best parameters are
immutable data, not mutable state on the Run.

In [ ]:
optimization = run.optimize(view, optimization_spec)
optimization.best.parameters
optimization.show()

[Previous](04_evaluate_quantity.qmd) · [Course
map](../../docs/index.qmd) · [Next: report and
resolve](06_report_resolve.qmd) · [Concept: units and
optimization](../../docs/concepts/units-parameters-and-optimization.qmd#parameter-vs-optimization-variable)